# Notebook 02: Q&A Dataset Generation

### Reads the cleaned article corpus from Notebook 01 and generates a verified question/answer test set using Gemini (free tier)
### Output: data/qa_dataset.json

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Install dependencies

In [2]:
!pip install google-genai -q

## Imports

In [3]:
import os
import json
import time
import random

from google import genai
from google.genai import types
from google.colab import userdata

## Config

In [16]:

API_KEY = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=API_KEY)

MODEL = "gemini-3.5-flash-lite"

DATA_DIR = "/content/drive/MyDrive/thesis-rag-cag/data"
WEEK_FILES = ["week1.json", "week2.json", "week3.json"]  # smallest to largest, cumulative
OUTPUT_PATH = os.path.join(DATA_DIR, "qa_dataset.json")
PROGRESS_PATH = os.path.join(DATA_DIR, "qa_progress.jsonl")  # one JSON object per line

PAIRS_PER_ARTICLE = 2  # one factual, one reasoning

# Retry/backoff for a single call that errors out (rare: network blip, etc).
MAX_RETRIES = 6
BASE_BACKOFF_SECONDS = 3
MAX_BACKOFF_SECONDS = 45
# cap so late retries don't wait forever
# Proactive pacing: wait at least this long between calls so we stay under
# the free-tier rate limit instead of hitting it and paying for it in long
# reactive backoffs. 15 requests/minute -> 1 call every 4 seconds; we pad
# a little for safety.
MIN_SECONDS_BETWEEN_CALLS = 4.5
_last_call_time = 0.0

GENERATE_SCHEMA = {
    "type": "object",
    "properties": {
        "pairs": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "question": {"type": "string"},
                    "answer": {"type": "string"},
                    "type": {"type": "string", "enum": ["factual", "reasoning"]},
                },
                "required": ["question", "answer", "type"],
            },
        }
    },
    "required": ["pairs"],
}

# One verify call now checks ALL pairs for an article at once (instead of
# one call per pair) — this is what cuts total API calls per article from
# 3 down to 2.
VERIFY_SCHEMA = {
    "type": "object",
    "properties": {
        "verdicts": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "supported": {"type": "boolean"},
                    "reason": {"type": "string"},
                },
                "required": ["supported", "reason"],
            },
        }
    },
    "required": ["verdicts"],
}

GENERATE_PROMPT = """You will be given a single news article. Write exactly {n} question/answer \
pairs about it:

1. One FACTUAL pair — a question whose answer is a specific fact stated directly in the \
article (a number, name, date, or event).
2. One REASONING pair — a question that requires connecting two or more pieces of \
information WITHIN this same article (e.g. a cause and its stated effect, or a comparison \
the article itself makes). Do not require any outside knowledge.

Rules:
- Use ONLY information explicitly stated in the article text below. Do not use outside \
knowledge, do not guess, do not infer anything the article does not say.
- Each answer must be short and precise (a phrase or one sentence), not a copy of the \
whole article.
- If the article does not contain enough concrete, verifiable information to write both \
kinds of questions, write what you honestly can and leave the rest out rather than \
inventing facts.

Article title: {title}

Article text:
{text}
"""

VERIFY_PROMPT = """You are fact-checking a list of question/answer pairs against a source \
article. You did NOT write these pairs — treat them with suspicion.

Article text:
{text}

Pairs to check (JSON):
{pairs_json}

For EACH pair, in the same order, decide: is the answer DIRECTLY and EXPLICITLY supported \
by the article text above? Mark "supported": false if the answer requires outside \
knowledge, is only loosely implied, contains a fact not in the text, or is simply wrong \
given the text. Be strict. Return one verdict per pair, in the same order as given.
"""

## Load articles and tag each with its earliest batch

In [17]:

def load_batches():
    batches = {}
    for filename in WEEK_FILES:
        path = os.path.join(DATA_DIR, filename)
        with open(path) as f:
            batches[filename] = json.load(f)
    return batches


batches = load_batches()

# week1 ⊂ week2 ⊂ week3, so the earliest file an article's id appears in
# is its "earliest batch" for later RAG-vs-CAG comparisons.
earliest_batch_by_id = {}
for filename in WEEK_FILES:  # smallest to largest — first assignment wins
    for article in batches[filename]:
        if article["id"] not in earliest_batch_by_id:
            earliest_batch_by_id[article["id"]] = filename.replace(".json", "")

all_articles = {a["id"]: a for a in batches["week3.json"]}  # week3 is the full set
print(f"Loaded {len(all_articles)} articles total.")

Loaded 369 articles total.


## Resume support — load whatever progress already exists

In [18]:
def load_progress():
    """Returns (qa_pairs, processed_article_ids). Safe to call on a fresh run
    (no file yet) or a resumed one (file has earlier results)."""
    qa_pairs = []
    processed_ids = set()
    if os.path.exists(PROGRESS_PATH):
        with open(PROGRESS_PATH) as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                record = json.loads(line)
                processed_ids.add(record["article_id"])
                qa_pairs.extend(record["qa_pairs"])  # may be empty list for a skipped article
    return qa_pairs, processed_ids


qa_dataset, already_processed_ids = load_progress()
if already_processed_ids:
    print(f"Resuming: {len(already_processed_ids)} articles already processed, "
          f"{len(qa_dataset)} Q&A pairs already saved.")

progress_file = open(PROGRESS_PATH, "a")  # append mode — never overwrite prior progress


def record_progress(article_id, pairs_for_this_article):
    """Appends one line to the progress file and flushes immediately, so
    a disconnect right after this call still keeps the result."""
    progress_file.write(json.dumps({
        "article_id": article_id,
        "qa_pairs": pairs_for_this_article,
    }) + "\n")
    progress_file.flush()
    os.fsync(progress_file.fileno())

Resuming: 290 articles already processed, 527 Q&A pairs already saved.


## Gemini call helper — paced + retried

In [19]:
class QuotaExhausted(Exception):
    """Raised when the API reports a DAILY quota is used up — retrying won't
    help for a long time (often an hour or more), so we stop immediately
    instead of burning through MAX_RETRIES pointlessly."""
    pass


def call_gemini(prompt, schema):
    global _last_call_time

    for attempt in range(MAX_RETRIES):
        # Proactive pacing: never call faster than our minimum interval.
        elapsed = time.time() - _last_call_time
        if elapsed < MIN_SECONDS_BETWEEN_CALLS:
            time.sleep(MIN_SECONDS_BETWEEN_CALLS - elapsed)

        try:
            response = client.models.generate_content(
                model=MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    response_schema=schema,
                    temperature=0.3,
                ),
            )
            _last_call_time = time.time()
            return json.loads(response.text)
        except Exception as e:
            _last_call_time = time.time()
            error_text = str(e)

            # A daily quota error won't resolve itself within seconds — don't
            # waste retries on it, bail out immediately so the caller can
            # stop the whole run cleanly.
            if "RESOURCE_EXHAUSTED" in error_text and "PerDay" in error_text:
                print(f"    Daily quota exhausted for model '{MODEL}': {error_text}")
                raise QuotaExhausted(error_text)

            wait = BASE_BACKOFF_SECONDS * (2 ** attempt) + random.uniform(0, 1)
            print(f"    Gemini call failed ({e}); retrying in {wait:.1f}s "
                  f"[attempt {attempt + 1}/{MAX_RETRIES}]")
            time.sleep(wait)

    print("    Giving up on this call after max retries (non-quota error).")
    return None

## Generate + verify, one article at a time

In [20]:
skipped_articles = []
dropped_unverified = 0
stopped_early = False

article_items = [
    (aid, a) for aid, a in all_articles.items() if aid not in already_processed_ids
]
print(f"{len(article_items)} articles left to process "
      f"({len(already_processed_ids)} already done).")

for i, (article_id, article) in enumerate(article_items):
    print(f"[{i + 1}/{len(article_items)}] {article_id}: {article['title'][:70]}")

    gen_prompt = GENERATE_PROMPT.format(
        n=PAIRS_PER_ARTICLE, title=article["title"], text=article["text"]
    )
    try:
        generated = call_gemini(gen_prompt, GENERATE_SCHEMA)
    except QuotaExhausted:
        stopped_early = True
        break

    if generated is None:
        # The call failed repeatedly for a non-quota reason (e.g. a brief
        # outage). Don't mark this article as processed — stop the run so
        # nothing gets silently skipped forever; re-running later retries it.
        print("    Stopping run: repeated non-quota API failures. "
              "Re-run this notebook to continue from this article.")
        stopped_early = True
        break

    if not generated.get("pairs"):
        # The call succeeded, but Gemini genuinely found nothing usable in
        # this article — a legitimate skip, safe to record permanently.
        skipped_articles.append(article_id)
        record_progress(article_id, [])
        continue

    pairs = generated["pairs"]
    verify_prompt = VERIFY_PROMPT.format(
        text=article["text"],
        pairs_json=json.dumps([{"question": p["question"], "answer": p["answer"]} for p in pairs]),
    )
    try:
        verification = call_gemini(verify_prompt, VERIFY_SCHEMA)
    except QuotaExhausted:
        stopped_early = True
        break

    if verification is None:
        print("    Stopping run: repeated non-quota API failures during verification. "
              "Re-run this notebook to continue from this article.")
        stopped_early = True
        break

    verdicts = verification.get("verdicts", [])

    pairs_for_this_article = []
    for pair, verdict in zip(pairs, verdicts):
        if not verdict or not verdict.get("supported"):
            dropped_unverified += 1
            continue
        pairs_for_this_article.append({
            "article_id": article_id,
            "earliest_batch": earliest_batch_by_id[article_id],
            "question": pair["question"],
            "answer": pair["answer"],
            "question_type": pair["type"],
            "source_url": article["url"],
            "source_title": article["title"],
        })

    # Any pairs Gemini generated but didn't return a matching verdict for
    # (a malformed response) are dropped rather than kept unverified.
    dropped_unverified += max(0, len(pairs) - len(verdicts))

    qa_dataset.extend(pairs_for_this_article)
    record_progress(article_id, pairs_for_this_article)

progress_file.close()

print(f"\nArticles skipped entirely (no usable content): {len(skipped_articles)}")
print(f"Q&A pairs dropped at verification: {dropped_unverified}")
print(f"Q&A pairs generated so far (this run + earlier resumed progress): {len(qa_dataset)}")

if stopped_early:
    done_count = len(already_processed_ids) + i  # i = index of the article that stopped us
    print(f"\n*** Stopped early after {done_count}/{len(all_articles)} articles — "
          f"this is expected on a quota limit, NOT an error. ***")
    print("Just re-run this notebook (from the top if your runtime restarted, or just the "
          "Config cell + this loop if it's still the same session) once the quota window "
          "has passed — it will automatically continue from here.")
else:
    print("\nAll articles processed.")

79 articles left to process (290 already done).
[1/79] article_0290: Bitcoin miners face widening AI financing costs as lenders scrutinize 
[2/79] article_0291: Broadcom CEO compares AI investors to gamblers: ‘They are in a casino’
    Gemini call failed (503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}); retrying in 3.1s [attempt 1/6]
    Gemini call failed (503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}); retrying in 3.9s [attempt 1/6]
[3/79] article_0292: Sam Altman Is ‘Very Uncomfortable’ With People Treating AI Like a Reli
[4/79] article_0293: Muse Wonder Puts Meta Back at the AI Frontier
[5/79] article_0294: Bitcoin miners need roughly $50 billion to finish their AI pivot
    Gemini call fai

## Save

In [21]:
for i, qa in enumerate(qa_dataset):
    qa["qa_id"] = f"qa_{i:04d}"

with open(OUTPUT_PATH, "w") as f:
    json.dump(qa_dataset, f, indent=2)

print(f"\nSaved {len(qa_dataset)} verified Q&A pairs to {OUTPUT_PATH}")

by_batch = {}
for qa in qa_dataset:
    by_batch[qa["earliest_batch"]] = by_batch.get(qa["earliest_batch"], 0) + 1
print("Breakdown by earliest batch (how many questions are answerable starting at each stage):")
for batch_name in ["week1", "week2", "week3"]:
    print(f"  {batch_name}: {by_batch.get(batch_name, 0)}")

if stopped_early:
    print("\nNote: this save only reflects progress up to where the run stopped. "
          "Re-run the notebook to finish, then re-run this save cell for the complete set.")
else:
    print("\nDone. Next step: Notebook 03 — build the RAG and CAG pipelines and run them "
          "against data/qa_dataset.json at each weekly batch size.")


Saved 670 verified Q&A pairs to /content/drive/MyDrive/thesis-rag-cag/data/qa_dataset.json
Breakdown by earliest batch (how many questions are answerable starting at each stage):
  week1: 54
  week2: 81
  week3: 535

Done. Next step: Notebook 03 — build the RAG and CAG pipelines and run them against data/qa_dataset.json at each weekly batch size.
